# Блок 7. Основы искусственного интеллекта и машинного обучения
# Занятие 5. Классификация и Logistic Regression

## Что такое классификация

Если регрессия предсказывает число, то классификация предсказывает категорию.

Примеры:

- купит клиент или нет;
- спам или не спам;
- болен или здоров;
- сдаст студент экзамен или нет.

## Бинарная классификация

Самый простой случай:

- 0 = Нет
- 1 = Да

## Logistic Regression

Несмотря на название, это алгоритм классификации.

Модель вычисляет вероятность принадлежности объекта к классу.

## Датасет занятия

Используем датасет Breast Cancer из Scikit-Learn.

Задача модели:

- 0 = злокачественное образование
- 1 = доброкачественное образование

## Связь с дипломом

Классификация используется в:

- HR;
- медицине;
- финансах;
- продажах;
- маркетинге.


Сначала изучите SOLVED-ноутбук вместе с преподавателем, затем выполните TODO самостоятельно.

## Ячейка 1. Загрузка датасета

### Алгоритм
Загружаем встроенный датасет классификации.

In [62]:
import pandas as pd
from IPython.display import display

url = "../../data/daily_table.csv"
df: pd.DataFrame = pd.read_csv(url)



df["weight_diff"] = df["weight"] - df["start_weight"]
target_col = "weight_gain"
feature_cols = ['energy_kcal', 'protein_g', 'fat_g', 'carbohydrates_g', 'fiber_g', 'sugars_g', 'saturated_fat_g',
                'cholesterol_mg', 'sodium_mg', 'potassium_mg', 'water_g']
df[target_col] = (df["weight_diff"] > 0).astype("int8")


print(df.shape)
display(df.head())

assert target_col in df.columns
assert len(df) > 1000
assert df.shape[1] == 19

(5249, 19)


,id,date,meals,products,start_weight,weight,energy_kcal,protein_g,fat_g,carbohydrates_g,fiber_g,sugars_g,saturated_fat_g,cholesterol_mg,sodium_mg,potassium_mg,water_g,weight_diff,weight_gain
0,0060b6899180e194092d60a0e50aeea1,2019/2/22,3,7,69.0,68.3,1418.0,65.12,37.88,210.64,19.5,32.23,5.131,44.0,1738.0,1032.0,299.40,-0.7,0
1,0060b6899180e194092d60a0e50aeea1,2019/2/23,3,12,68.3,68.8,2301.0,63.65,92.39,309.36,24.8,66.61,13.076,82.0,3655.0,1925.0,618.95,0.5,1
2,0060b6899180e194092d60a0e50aeea1,2019/2/24,3,13,68.8,68.2,2684.0,87.26,69.32,434.98,28.1,130.80,17.871,161.0,7483.0,2572.0,667.44,-0.6,0
3,0060b6899180e194092d60a0e50aeea1,2019/2/25,3,6,68.2,68.2,1387.0,46.61,66.30,163.96,11.6,3.06,24.028,80.0,1516.0,2041.0,312.80,0.0,0
4,0060b6899180e194092d60a0e50aeea1,2019/2/26,3,6,68.2,68.2,1466.0,25.95,78.55,174.22,10.7,48.69,21.807,6.0,2094.0,986.0,312.31,0.0,0


## Ячейка 2. Изучение данных

### Алгоритм
Смотрим признаки и классы.

In [63]:
X: pd.DataFrame = pd.DataFrame(df, columns=feature_cols)
y: pd.Series = pd.Series(df[target_col])


assert len(X) == len(y)

## Ячейка 3. Распределение классов

### Алгоритм
Считаем объекты каждого класса.

In [64]:
class_counts = y.value_counts()

print(class_counts)

assert class_counts.sum() == len(y)

weight_gain
0    4022
1    1227
Name: count, dtype: int64


## Ячейка 4. Train/Test Split

### Алгоритм
Разделяем данные.

In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

assert len(X_train) > len(X_test)

## Ячейка 5. Масштабирование

### Алгоритм
Подготавливаем признаки.

In [66]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

assert X_train_scaled.shape[0] == len(X_train)

## Ячейка 6. Обучение Logistic Regression

### Алгоритм
Строим модель.

In [67]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000, class_weight="balanced")

model.fit(X_train_scaled, y_train)

assert model is not None

## Ячейка 7. Предсказание классов

### Алгоритм
Получаем прогноз.

In [68]:
predictions = model.predict(X_test_scaled)
print(predictions[:10])

assert len(predictions) == len(y_test)

[1 0 1 1 0 1 0 0 0 0]


## Ячейка 8. Вероятности

### Алгоритм
Получаем вероятность классов.

In [69]:
probabilities = model.predict_proba(X_test_scaled)

print(probabilities[:5])

assert probabilities.shape[1] == 2

[[0.47419427 0.52580573]
 [0.52780644 0.47219356]
 [0.47162537 0.52837463]
 [0.46919407 0.53080593]
 [0.52624484 0.47375516]]


## Ячейка 9. Метрики классификации

### Алгоритм
Считаем Accuracy, Precision, Recall, F1.

In [71]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy", accuracy)
print("Precision", precision)
print("Recall", recall)
print("F1", f1)

assert accuracy > 0.1

Accuracy 0.518095238095238
Precision 0.25
Recall 0.512
F1 0.3359580052493438


## Ячейка 10. Мини-проект

### Алгоритм
Формируем итоговый отчет.

In [ ]:
# TODO: выполните самостоятельно задание ячейки 10